# Segmentación de Documentos con U-Net y ResNet50-U-Net

Clasificación semántica píxel a píxel de documentos manuscritos/impresos en 6 categorías:

| Clase | Índice | Color |
|-------|--------|-------|
| Fondo | 0 | Negro |
| Texto principal | 1 | Azul |
| Imágenes / Ilustraciones | 2 | Verde |
| Separadores / Líneas | 3 | Amarillo |
| Encabezados | 4 | Rojo |
| Letras capitulares | 5 | Magenta |

> **Arquitecturas implementadas:** U-Net estándar y ResNet50-U-Net con encoder preentrenado en ImageNet (PyTorch + segmentation_models_pytorch)


## 1. Instalación e Importación de Librerías


In [ ]:
# Instalar dependencias (ejecutar una sola vez)
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
%pip install segmentation-models-pytorch albumentations opencv-python matplotlib numpy tqdm -q


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# ── Dispositivo ──────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Clases de Segmentación y Paleta de Colores


In [ ]:
# ── Definición de clases ─────────────────────────────────────────────────────
CLASSES = {
    0: "Fondo",
    1: "Texto principal",
    2: "Imagen / Ilustración",
    3: "Separador / Línea",
    4: "Encabezado",
    5: "Letra capitular",
}
NUM_CLASSES = len(CLASSES)

# Paleta RGB para visualización
PALETTE = np.array([
    [0,   0,   0  ],   # 0 - Fondo          → Negro
    [70,  130, 180],   # 1 - Texto principal → Azul acero
    [60,  179, 113],   # 2 - Imagen          → Verde mar
    [255, 215, 0  ],   # 3 - Separador       → Dorado
    [220, 50,  47 ],   # 4 - Encabezado      → Rojo
    [200, 50,  200],   # 5 - Letra capitular → Magenta
], dtype=np.uint8)


def colorize_mask(mask: np.ndarray) -> np.ndarray:
    """Convierte una máscara de índices (H,W) a imagen RGB usando PALETTE."""
    rgb = PALETTE[mask.astype(np.int32)]
    return rgb


def show_legend(ax=None):
    patches = [
        mpatches.Patch(color=PALETTE[i] / 255.0, label=f"{i}: {CLASSES[i]}")
        for i in range(NUM_CLASSES)
    ]
    if ax is None:
        plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left")
    else:
        ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left")


# Verificar paleta
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    ax.imshow([[PALETTE[i]]])
    ax.set_title(f"{i}\n{CLASSES[i]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Paleta de clases de segmentación", fontsize=11, y=1.05)
plt.tight_layout()
plt.show()


## 3. Dataset — Carga y Exploración

El dataset espera la siguiente estructura de carpetas:
```
data/
  images/       ← imágenes originales (PNG/JPG/TIF)
  masks/        ← máscaras de segmentación (PNG, 1 canal, valores 0-5)
```
Si aún no tienes datos etiquetados, la siguiente celda **genera documentos sintéticos** para probar el pipeline completo.


In [ ]:
def generate_synthetic_document(h=768, w=512, seed=None):
    """
    Genera un documento sintético con regiones bien definidas
    junto con su máscara ground-truth de segmentación.
    Retorna (imagen BGR uint8, máscara índices uint8).
    """
    rng = np.random.default_rng(seed)
    img  = np.ones((h, w, 3), dtype=np.uint8) * 245  # fondo crema
    mask = np.zeros((h, w), dtype=np.uint8)            # todo fondo

    # ── Encabezado ──────────────────────────────────────────────────────────
    header_h = rng.integers(40, 80)
    img[:header_h]  = rng.integers(210, 235, (1, 3)).astype(np.uint8)
    mask[:header_h] = 4  # encabezado
    # texto simulado en encabezado
    for _ in range(rng.integers(2, 5)):
        x = rng.integers(10, w - 80)
        cv2.rectangle(img, (x, 8), (x + rng.integers(40, 120), header_h - 8),
                      (80, 80, 80), -1)
        mask[8:header_h - 8, x:x + rng.integers(40, 120)] = 4

    # ── Separador bajo encabezado ────────────────────────────────────────────
    sep_y = header_h + 6
    cv2.line(img, (10, sep_y), (w - 10, sep_y), (100, 100, 100), 2)
    mask[sep_y - 2:sep_y + 2, 10:w - 10] = 3

    # ── Letra capitular ──────────────────────────────────────────────────────
    cap_x, cap_y = 10, sep_y + 15
    cap_s = rng.integers(45, 70)
    cv2.rectangle(img, (cap_x, cap_y), (cap_x + cap_s, cap_y + cap_s),
                  (150, 50, 200), -1)
    mask[cap_y:cap_y + cap_s, cap_x:cap_x + cap_s] = 5

    # ── Texto principal (líneas simuladas) ───────────────────────────────────
    y = cap_y
    txt_x0 = cap_x + cap_s + 8
    line_h = rng.integers(10, 16)
    while y + line_h < h - 60:
        x0 = txt_x0 if y < cap_y + cap_s else 10
        x1 = w - rng.integers(10, 40)
        alpha = rng.integers(60, 140)
        color = (int(alpha), int(alpha), int(alpha))
        cv2.rectangle(img, (x0, y + 2), (x1, y + line_h - 2), color, -1)
        mask[y + 2:y + line_h - 2, x0:x1] = 1
        y += line_h + rng.integers(3, 8)
        txt_x0 = 10  # a partir de la 2ª línea va al margen

    # ── Imagen / ilustración ─────────────────────────────────────────────────
    il_h = rng.integers(80, 140)
    il_w = rng.integers(100, 200)
    il_x = rng.integers(10, w - il_w - 10)
    il_y = h - il_h - 50
    img[il_y:il_y + il_h, il_x:il_x + il_w] = rng.integers(
        100, 200, (il_h, il_w, 3), dtype=np.uint8)
    mask[il_y:il_y + il_h, il_x:il_x + il_w] = 2

    # ── Separador antes del pie ──────────────────────────────────────────────
    foot_sep = h - 45
    cv2.line(img, (10, foot_sep), (w - 10, foot_sep), (100, 100, 100), 1)
    mask[foot_sep - 1:foot_sep + 1, 10:w - 10] = 3

    return img, mask


# Generar dataset sintético en disco ─────────────────────────────────────────
DATA_DIR   = Path("data")
IMG_DIR    = DATA_DIR / "images"
MASK_DIR   = DATA_DIR / "masks"
IMG_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 60
for i in range(N_SAMPLES):
    img, mask = generate_synthetic_document(seed=i)
    cv2.imwrite(str(IMG_DIR  / f"doc_{i:04d}.png"), img)
    cv2.imwrite(str(MASK_DIR / f"doc_{i:04d}.png"), mask)  # valores 0-5

print(f"✅  Generados {N_SAMPLES} documentos sintéticos en '{DATA_DIR.resolve()}'")


In [ ]:
# Visualizar 4 muestras con sus máscaras ────────────────────────────────────
img_paths  = sorted(IMG_DIR.glob("*.png"))
mask_paths = sorted(MASK_DIR.glob("*.png"))

fig, axes = plt.subplots(4, 3, figsize=(13, 18))
indices = [0, 5, 15, 30]
for row, idx in enumerate(indices):
    img  = cv2.cvtColor(cv2.imread(str(img_paths[idx])),  cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
    colored = colorize_mask(mask)
    # superposición 50%
    overlay = (img * 0.5 + colored * 0.5).astype(np.uint8)

    axes[row, 0].imshow(img);      axes[row, 0].set_title(f"Imagen #{idx}")
    axes[row, 1].imshow(colored);  axes[row, 1].set_title("Máscara GT")
    axes[row, 2].imshow(overlay);  axes[row, 2].set_title("Superposición")
    for ax in axes[row]: ax.axis("off")

# Leyenda global
patches = [mpatches.Patch(color=PALETTE[i]/255, label=f"{i}: {CLASSES[i]}")
           for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.01))
plt.suptitle("Muestras del dataset", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Distribución de clases ──────────────────────────────────────────────────────
all_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).flatten()
             for p in mask_paths[:20]]
flat = np.concatenate(all_masks)
counts = [(flat == c).sum() for c in range(NUM_CLASSES)]
total  = sum(counts)

print("\nDistribución de píxeles (primeras 20 imágenes):")
for i, (name, cnt) in enumerate(zip(CLASSES.values(), counts)):
    bar = "█" * int(cnt / total * 40)
    print(f"  {i} {name:<22} {cnt:>10,}  {cnt/total*100:5.1f}%  {bar}")


## 4. Dataset PyTorch y Augmentaciones


In [ ]:
IMG_SIZE = 512   # redimensionar a 512×512
BATCH_SIZE = 4
NUM_WORKERS = 0  # en Windows usar 0 para evitar problemas con multiprocessing

# ── Augmentaciones ────────────────────────────────────────────────────────────
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.3),
    A.RandomRotate90(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=10, p=0.4,
                       border_mode=cv2.BORDER_CONSTANT),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.GaussianBlur(blur_limit=3),
        A.GaussNoise(var_limit=(5, 30)),
    ], p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


# ── Dataset ───────────────────────────────────────────────────────────────────
class DocumentSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(str(self.img_paths[idx])),
                            cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, NUM_CLASSES - 1)

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out["image"]          # (3, H, W) float32
            mask = out["mask"].long()    # (H, W) int64
        return img, mask


# ── Splits ────────────────────────────────────────────────────────────────────
img_paths_all  = sorted(IMG_DIR.glob("*.png"))
mask_paths_all = sorted(MASK_DIR.glob("*.png"))

n = len(img_paths_all)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
n_test  = n - n_train - n_val

# Mezcla reproducible
rng_split = np.random.default_rng(42)
idxs = rng_split.permutation(n)
tr_i, va_i, te_i = (idxs[:n_train],
                    idxs[n_train:n_train+n_val],
                    idxs[n_train+n_val:])

def subset(indices, tf):
    ip = [img_paths_all[i]  for i in indices]
    mp = [mask_paths_all[i] for i in indices]
    return DocumentSegDataset(ip, mp, transform=tf)

train_ds = subset(tr_i, train_transforms)
val_ds   = subset(va_i, val_transforms)
test_ds  = subset(te_i, val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=1,          shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Batches train: {len(train_loader)} | val: {len(val_loader)}")

# Verificar un batch ──────────────────────────────────────────────────────────
imgs, masks = next(iter(train_loader))
print(f"\nBatch imagen : {imgs.shape}  dtype={imgs.dtype}")
print(f"Batch máscara: {masks.shape}  dtype={masks.dtype}  valores únicos={masks.unique().tolist()}")


## 5. Arquitectura U-Net estándar (desde cero)

Encoder con bloques de convolución doble + MaxPool, decoder simétrico con `ConvTranspose2d` y **skip connections**.

```
Entrada (3, 512, 512)
  └─ Encoder: [64→128→256→512→1024]
       └─ Bottleneck
  └─ Decoder: [512→256→128→64]
       └─ Salida: (NUM_CLASSES, 512, 512)  ← softmax por clase
```


In [ ]:
class DoubleConv(nn.Module):
    """Conv → BN → ReLU  ×2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, features=(64, 128, 256, 512)):
        super().__init__()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        # ── Encoder ──────────────────────────────────────────────────────────
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        # ── Bottleneck ────────────────────────────────────────────────────────
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # ── Decoder ───────────────────────────────────────────────────────────
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.ups.append(DoubleConv(f * 2, f))

        self.head = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape != skip.shape:
                x = TF.resize(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x)

        return self.head(x)


# ── Resumen ───────────────────────────────────────────────────────────────────
unet = UNet(num_classes=NUM_CLASSES).to(DEVICE)
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out   = unet(dummy)
print(f"UNet — entrada: {dummy.shape}  →  salida: {out.shape}")
n_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")
del dummy, out


## 6. Arquitectura ResNet50-U-Net (encoder preentrenado)

Se usa `segmentation_models_pytorch` para construir un **U-Net con backbone ResNet-50** inicializado con pesos ImageNet. El decoder conecta los mapas de características intermedios de ResNet50 con skip connections, igual que el U-Net clásico pero con un encoder mucho más potente.


In [ ]:
resnet_unet = smp.Unet(
    encoder_name    = "resnet50",        # backbone ResNet-50
    encoder_weights = "imagenet",        # pesos preentrenados
    in_channels     = 3,
    classes         = NUM_CLASSES,
    activation      = None,              # aplicaremos softmax en la loss
).to(DEVICE)

dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out   = resnet_unet(dummy)
print(f"ResNet50-UNet — entrada: {dummy.shape}  →  salida: {out.shape}")
n_params = sum(p.numel() for p in resnet_unet.parameters() if p.requires_grad)
enc_params = sum(p.numel() for p in resnet_unet.encoder.parameters() if p.requires_grad)
dec_params = n_params - enc_params
print(f"Parámetros totales: {n_params:,}")
print(f"  Encoder (ResNet50): {enc_params:,}")
print(f"  Decoder:            {dec_params:,}")
del dummy, out


## 7. Función de Pérdida y Métricas

Se combina **Cross-Entropy** con **Dice Loss** para manejar el desequilibrio de clases (el fondo domina):

$$\mathcal{L} = \mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{Dice}$$

$$\mathcal{L}_{Dice} = 1 - \frac{2\sum_i p_i g_i + \varepsilon}{\sum_i p_i + \sum_i g_i + \varepsilon}$$


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, num_classes, eps=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.eps = eps

    def forward(self, logits, targets):
        """
        logits : (B, C, H, W) — sin softmax
        targets: (B, H, W)    — índices de clase long
        """
        probs = torch.softmax(logits, dim=1)
        # one-hot targets: (B, C, H, W)
        tgt_oh = torch.zeros_like(probs)
        tgt_oh.scatter_(1, targets.unsqueeze(1), 1.0)

        dims = (0, 2, 3)
        inter  = (probs * tgt_oh).sum(dims)
        cardin = (probs + tgt_oh).sum(dims)
        dice   = (2 * inter + self.eps) / (cardin + self.eps)
        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, num_classes, dice_weight=0.5,
                 class_weights=None):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes)
        self.lam  = dice_weight

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.lam * self.dice(logits, targets)


def mean_iou(logits, targets, num_classes):
    """Calcula mIoU sobre un batch."""
    preds = logits.argmax(dim=1)
    ious = []
    for cls in range(num_classes):
        pred_c = preds == cls
        tgt_c  = targets == cls
        inter  = (pred_c & tgt_c).sum().float()
        union  = (pred_c | tgt_c).sum().float()
        if union == 0:
            continue
        ious.append((inter / union).item())
    return np.mean(ious) if ious else 0.0


def pixel_accuracy(logits, targets):
    preds   = logits.argmax(dim=1)
    correct = (preds == targets).sum().float()
    return (correct / targets.numel()).item()


print("✅  Loss y métricas definidas")
# Prueba rápida
dummy_logits  = torch.randn(2, NUM_CLASSES, 64, 64)
dummy_targets = torch.randint(0, NUM_CLASSES, (2, 64, 64))
loss_fn = CombinedLoss(NUM_CLASSES)
print(f"   Loss de prueba: {loss_fn(dummy_logits, dummy_targets):.4f}")
print(f"   mIoU de prueba: {mean_iou(dummy_logits, dummy_targets, NUM_CLASSES):.4f}")
print(f"   PixAcc de prueba: {pixel_accuracy(dummy_logits, dummy_targets):.4f}")


## 8. Entrenamiento

Selecciona el modelo a entrenar cambiando `MODEL_NAME = "unet"` o `"resnet50_unet"`.


In [ ]:
# ─── Configuración ────────────────────────────────────────────────────────────
MODEL_NAME  = "resnet50_unet"   # "unet"  o  "resnet50_unet"
EPOCHS      = 30
LR          = 1e-4
CHECKPOINT  = Path(f"checkpoints/{MODEL_NAME}_best.pt")
CHECKPOINT.parent.mkdir(exist_ok=True)

model = resnet_unet if MODEL_NAME == "resnet50_unet" else UNet(num_classes=NUM_CLASSES).to(DEVICE)

optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", patience=5, factor=0.5, verbose=True)
loss_fn    = CombinedLoss(NUM_CLASSES, dice_weight=0.5)

# ─── Bucle de entrenamiento ───────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "val_miou": [], "val_acc": []}
best_miou = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [train]",
                            leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = loss_fn(out, masks)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    # ── Validación ────────────────────────────────────────────────────────────
    model.eval()
    va_loss = va_miou = va_acc = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            out     = model(imgs)
            va_loss += loss_fn(out, masks).item()
            va_miou += mean_iou(out, masks, NUM_CLASSES)
            va_acc  += pixel_accuracy(out, masks)
    va_loss /= len(val_loader)
    va_miou /= len(val_loader)
    va_acc  /= len(val_loader)

    scheduler.step(va_miou)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_miou"].append(va_miou)
    history["val_acc"].append(va_acc)

    marker = ""
    if va_miou > best_miou:
        best_miou = va_miou
        torch.save(model.state_dict(), CHECKPOINT)
        marker = "  ← mejor"

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"loss {tr_loss:.4f} / {va_loss:.4f}  "
          f"mIoU {va_miou:.4f}  "
          f"acc {va_acc:.4f}{marker}")

print(f"\n🏆  Mejor mIoU en validación: {best_miou:.4f}")


In [ ]:
# ─── Curvas de entrenamiento ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train")
axes[0].plot(ep, history["val_loss"],   label="Val")
axes[0].set_title("Loss (CE + Dice)"); axes[0].legend(); axes[0].set_xlabel("Época")

axes[1].plot(ep, history["val_miou"], color="steelblue")
axes[1].set_title("mIoU — Validación"); axes[1].set_xlabel("Época")

axes[2].plot(ep, history["val_acc"], color="seagreen")
axes[2].set_title("Pixel Accuracy — Validación"); axes[2].set_xlabel("Época")

plt.suptitle(f"Entrenamiento: {MODEL_NAME}", fontsize=13)
plt.tight_layout()
plt.show()


## 9. Evaluación e Inferencia en Test


In [ ]:
# ── Cargar el mejor checkpoint ────────────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()
print(f"Checkpoint cargado: {CHECKPOINT}")

# ── IoU por clase sobre test set ──────────────────────────────────────────────
iou_per_class = np.zeros(NUM_CLASSES)
iou_counts    = np.zeros(NUM_CLASSES)
all_acc       = []

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc="Evaluando test"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        out   = model(imgs)
        preds = out.argmax(dim=1)
        all_acc.append(pixel_accuracy(out, masks))

        for cls in range(NUM_CLASSES):
            pred_c = (preds == cls)
            tgt_c  = (masks == cls)
            inter  = (pred_c & tgt_c).sum().float().item()
            union  = (pred_c | tgt_c).sum().float().item()
            if union > 0:
                iou_per_class[cls] += inter / union
                iou_counts[cls]    += 1

mean_ious = np.where(iou_counts > 0, iou_per_class / iou_counts, np.nan)

print(f"\n{'Clase':<25} {'IoU':>8}")
print("─" * 35)
for i, (name, iou) in enumerate(zip(CLASSES.values(), mean_ious)):
    bar = "█" * int(iou * 20) if not np.isnan(iou) else ""
    val = f"{iou:.4f}" if not np.isnan(iou) else "  N/A "
    print(f"  {name:<23} {val}  {bar}")
print("─" * 35)
print(f"  {'mIoU':<23} {np.nanmean(mean_ious):.4f}")
print(f"  {'Pixel Accuracy':<23} {np.mean(all_acc):.4f}")


In [ ]:
def predict_document(model, image_path, device=DEVICE):
    """Carga una imagen y devuelve la máscara predicha (H,W) con índices de clase."""
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    tf  = val_transforms(image=img)["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tf)
    pred = logits.argmax(dim=1).squeeze().cpu().numpy()
    return img, pred


# ── Visualizar predicciones sobre 4 imágenes de test ─────────────────────────
test_img_paths = [test_ds.img_paths[i] for i in range(min(4, len(test_ds)))]

fig, axes = plt.subplots(len(test_img_paths), 4, figsize=(16, 4 * len(test_img_paths)))

for row, path in enumerate(test_img_paths):
    idx = test_ds.img_paths.index(path)
    gt_mask = cv2.imread(str(test_ds.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)

    img_orig, pred_mask = predict_document(model, path)

    # Redimensionar GT al tamaño de la predicción para comparar
    gt_resized = cv2.resize(gt_mask, (pred_mask.shape[1], pred_mask.shape[0]),
                            interpolation=cv2.INTER_NEAREST)
    img_resized = cv2.resize(img_orig, (pred_mask.shape[1], pred_mask.shape[0]))

    gt_color   = colorize_mask(gt_resized)
    pred_color = colorize_mask(pred_mask)
    overlay    = (img_resized * 0.5 + pred_color * 0.5).astype(np.uint8)

    for ax, (data, title) in zip(axes[row], [
        (img_resized,  "Imagen original"),
        (gt_color,     "GT"),
        (pred_color,   "Predicción"),
        (overlay,      "Superposición"),
    ]):
        ax.imshow(data)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

patches = [mpatches.Patch(color=PALETTE[i]/255, label=f"{i}: {CLASSES[i]}")
           for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle(f"Inferencia — {MODEL_NAME}", fontsize=13)
plt.tight_layout()
plt.show()


## 10. Exportar modelo para inferencia en el backend

Una vez satisfecho con el modelo, se puede exportar a **TorchScript** o **ONNX** para integrarlo en el backend FastAPI.


In [ ]:
model.eval()
dummy_input = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

# ── Opción A: TorchScript (recomendado para PyTorch puro) ─────────────────────
ts_path = Path(f"checkpoints/{MODEL_NAME}_scripted.pt")
try:
    scripted = torch.jit.trace(model, dummy_input)
    scripted.save(str(ts_path))
    print(f"✅  TorchScript guardado en: {ts_path}")
except Exception as e:
    print(f"⚠️  TorchScript falló ({e}), prueba con ONNX")

# ── Opción B: ONNX (compatible con otros frameworks) ─────────────────────────
onnx_path = Path(f"checkpoints/{MODEL_NAME}.onnx")
torch.onnx.export(
    model,
    dummy_input,
    str(onnx_path),
    opset_version      = 17,
    input_names        = ["image"],
    output_names       = ["logits"],
    dynamic_axes       = {"image":  {0: "batch", 2: "height", 3: "width"},
                          "logits": {0: "batch", 2: "height", 3: "width"}},
    do_constant_folding= True,
)
print(f"✅  ONNX guardado en: {onnx_path}")
print(f"\n📦  Archivos exportados:")
for f in Path("checkpoints").iterdir():
    size = f.stat().st_size / 1e6
    print(f"   {f.name:<40} {size:.1f} MB")
